In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import polars as pl
import gc
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [4]:
TIME_STEPS_MAX = 967
FILTER_GRANULARITY = 1
ITERATIONS = 6
MIN_TRAIN_SAMPLES = 1000
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEQ_LENGTH = 5
HIDDEN_SIZE = 32
NUM_LAYERS = 1
DROPOUT = 0.1
LEARNING_RATE = 0.001
BATCH_SIZE = 512
EPOCHS = 20
PATIENCE = 3

In [5]:
def feature_engineering(df):
    df_pl = pl.from_pandas(df)

    df_pl = df_pl.with_columns([
        (2 * np.pi * pl.col('time_id') / TIME_STEPS_MAX).sin().alias('sin_time'),
        (2 * np.pi * pl.col('time_id') / TIME_STEPS_MAX).cos().alias('cos_time')
    ])

    if 'y' in df_pl.columns:
        df_pl = df_pl.with_columns(
            pl.col('y').shift(1).over('symbol_id').fill_null(0).alias('y_lag_1')
        )
    else:
        df_pl = df_pl.with_columns(pl.lit(0.0).alias('y_lag_1'))

    df_pl = df_pl.fill_nan(0).fill_null(0)

    feat_cols = [f'f{i}' for i in range(26)] + ['sin_time', 'cos_time', 'y_lag_1']
    X_design = df_pl.select(feat_cols).to_numpy()
    XtX_inv = np.linalg.pinv(X_design.T @ X_design)
    M = X_design @ XtX_inv
    leverages = np.sum(M * X_design, axis=1)

    df_pl = df_pl.with_columns(pl.Series(name="leverage", values=leverages))

    final_df = df_pl.to_pandas()
    for col in final_df.columns:
        if final_df[col].dtype == 'float64':
            final_df[col] = final_df[col].astype('float32')

    return final_df, feat_cols + ['leverage']

In [6]:
def iterative_time_filtering(train_df, valid_df, features, iterations=3):
    train_df['time_group'] = (train_df['time_id'] // FILTER_GRANULARITY).astype(int)
    current_train = train_df.copy()

    model = Ridge(alpha=1.0)
    model.fit(current_train[features], current_train['y'])
    best_rmse = np.sqrt(mean_squared_error(valid_df['y'], model.predict(valid_df[features])))

    print(f"Initial CV RMSE: {best_rmse:.8f}")

    for i in range(iterations):
        groups = current_train['time_group'].unique()
        test_groups = np.random.choice(groups, size=min(20, len(groups)), replace=False)

        for group in test_groups:
            temp_train = current_train[current_train['time_group'] != group]

            if len(temp_train) < MIN_TRAIN_SAMPLES:
                continue

            model.fit(temp_train[features], temp_train['y'])
            temp_rmse = np.sqrt(mean_squared_error(valid_df['y'], model.predict(valid_df[features])))

            if temp_rmse < best_rmse:
                print(f"Iter {i+1}: Pruning group {group} | New RMSE: {temp_rmse:.8f}")
                best_rmse = temp_rmse
                current_train = temp_train.copy()

    return current_train

In [21]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(GRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size, 1)

        self.init_weights()

    def init_weights(self):
        for name, param in self.gru.named_parameters():
            if 'weight' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                nn.init.constant_(param.data, 0)

        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.constant_(self.fc.bias, 0)

    def forward(self, x):
        batch_size = x.size(0)

        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)

        out, _ = self.gru(x, h0)

        out = out[:, -1, :]

        out = self.dropout(out)

        out = self.fc(out)

        return out

def create_sequences(data, seq_length, features):
    sequences = []
    targets = []

    grouped = data.groupby('symbol_id')

    for symbol, group in grouped:
        group = group.sort_values('time_id')

        feature_values = group[features].values

        target_values = group['y'].values

        for i in range(len(group) - seq_length):
            seq = feature_values[i:i+seq_length]
            target = target_values[i+seq_length]
            sequences.append(seq)
            targets.append(target)

    return np.array(sequences, dtype=np.float32), np.array(targets, dtype=np.float32)

def train_gru_model(train_sequences, train_targets, val_sequences, val_targets, features):
    train_X = torch.FloatTensor(train_sequences).to(DEVICE)
    train_y = torch.FloatTensor(train_targets).to(DEVICE).unsqueeze(1)
    val_X = torch.FloatTensor(val_sequences).to(DEVICE)
    val_y = torch.FloatTensor(val_targets).to(DEVICE).unsqueeze(1)

    train_dataset = TensorDataset(train_X, train_y)
    val_dataset = TensorDataset(val_X, val_y)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    input_size = len(features)
    model = GRUModel(
        input_size=input_size,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    ).to(DEVICE)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch_X, batch_y in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                predictions = model(batch_X)
                loss = criterion(predictions, batch_y)
                val_loss += loss.item() * batch_X.size(0)

        val_loss /= len(val_loader.dataset)

        print(f'Epoch {epoch+1}: Train Loss = {train_loss:.8f}, Val Loss = {val_loss:.8f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_gru_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'Early stopping at epoch {epoch+1}')
                break

        scheduler.step(val_loss)

    model.load_state_dict(torch.load('best_gru_model.pth'))
    return model

def evaluate_gru_model(model, val_loader):
    model.eval()
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch_X, batch_y in tqdm(val_loader, desc='Evaluating'):
            predictions = model(batch_X)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(batch_y.cpu().numpy())

    all_predictions = np.concatenate(all_predictions, axis=0).flatten()
    all_targets = np.concatenate(all_targets, axis=0).flatten()

    return all_predictions, all_targets

def predict_gru_test(model, test_df, features, seq_length, batch_size=512):
    model.eval()
    all_predictions = []
    all_ids = []

    grouped_test = test_df.groupby('symbol_id')

    for symbol, group in tqdm(grouped_test, desc='Predicting test'):
        group = group.sort_values('time_id')
        symbol_ids = group['Id'].values

        feature_values = group[features].values
        symbol_sequences = []
        for i in range(len(group)):
            if i < seq_length - 1:
                seq = np.zeros((seq_length, len(features)))
                if i > 0:
                    seq[-i-1:] = feature_values[:i+1]
            else:
                seq = feature_values[i-seq_length+1:i+1]
            symbol_sequences.append(seq)

        symbol_sequences = np.array(symbol_sequences, dtype=np.float32)
        symbol_tensor = torch.FloatTensor(symbol_sequences).to(DEVICE)

        symbol_preds = []
        with torch.no_grad():
            num_batches = int(np.ceil(len(symbol_tensor) / batch_size))
            for i in range(num_batches):
                start_idx = i * batch_size
                end_idx = min((i + 1) * batch_size, len(symbol_tensor))
                batch = symbol_tensor[start_idx:end_idx]
                preds = model(batch).cpu().numpy().flatten()
                symbol_preds.extend(preds)

        all_predictions.extend(symbol_preds)
        all_ids.extend(symbol_ids)

    return pd.DataFrame({'Id': all_ids, 'prediction_gru': all_predictions})

In [22]:
data_raw = pd.read_csv('/content/drive/MyDrive/quadeye-market-data-prediction-challenge/train.csv')
test_raw = pd.read_csv('/content/drive/MyDrive/quadeye-market-data-prediction-challenge/test.csv')

In [23]:
print(f"Training data shape: {data_raw.shape}")
print(f"Test data shape: {test_raw.shape}")

Training data shape: (2017127, 31)
Test data shape: (2510026, 30)


In [24]:
data, features = feature_engineering(data_raw)
test, _ = feature_engineering(test_raw)

unique_dates = sorted(data['date_id'].unique())
split_date = unique_dates[int(len(unique_dates) * 0.8)]
train_df_raw = data[data['date_id'] < split_date].copy()
valid_df = data[data['date_id'] >= split_date].copy()

In [25]:
print(f"Train size: {len(train_df_raw):,}")
print(f"Valid size: {len(valid_df):,}")

Train size: 1,606,179
Valid size: 410,948


In [26]:
scaler = StandardScaler()
train_df_raw[features] = scaler.fit_transform(train_df_raw[features])
valid_df[features] = scaler.transform(valid_df[features])
test[features] = scaler.transform(test[features])

In [27]:
print("\niterative filtering")
optimized_train = iterative_time_filtering(train_df_raw, valid_df, features, iterations=ITERATIONS)


iterative filtering
Initial CV RMSE: 0.00224240
Iter 1: Pruning group 7 | New RMSE: 0.00224238
Iter 1: Pruning group 6 | New RMSE: 0.00224238
Iter 1: Pruning group 23 | New RMSE: 0.00224237
Iter 1: Pruning group 20 | New RMSE: 0.00224237
Iter 1: Pruning group 53 | New RMSE: 0.00224231
Iter 1: Pruning group 50 | New RMSE: 0.00224210
Iter 1: Pruning group 17 | New RMSE: 0.00224209
Iter 1: Pruning group 35 | New RMSE: 0.00224209
Iter 2: Pruning group 49 | New RMSE: 0.00224189
Iter 2: Pruning group 30 | New RMSE: 0.00224189
Iter 2: Pruning group 51 | New RMSE: 0.00224172
Iter 2: Pruning group 40 | New RMSE: 0.00224172
Iter 2: Pruning group 15 | New RMSE: 0.00224171
Iter 2: Pruning group 11 | New RMSE: 0.00224170
Iter 2: Pruning group 3 | New RMSE: 0.00224158
Iter 2: Pruning group 12 | New RMSE: 0.00224158
Iter 3: Pruning group 5 | New RMSE: 0.00224155
Iter 3: Pruning group 48 | New RMSE: 0.00224154
Iter 3: Pruning group 4 | New RMSE: 0.00224146
Iter 3: Pruning group 42 | New RMSE: 0.00224

In [28]:
final_model = Ridge(alpha=0.1, random_state=42)
final_model.fit(optimized_train[features], optimized_train['y'])

Ridge(alpha=0.1, random_state=42)

In [29]:
v_preds = final_model.predict(valid_df[features])
ridge_rmse = np.sqrt(mean_squared_error(valid_df['y'], v_preds))
print(f"\nRidge Model - Final Optimized Val RMSE: {ridge_rmse:.8f}")
test['prediction_ridge'] = final_model.predict(test[features])


Ridge Model - Final Optimized Val RMSE: 0.00224144


In [30]:
print("Creating training sequences")
train_sequences, train_targets = create_sequences(optimized_train, SEQ_LENGTH, features)
print("Creating validation sequences")
val_sequences, val_targets = create_sequences(valid_df, SEQ_LENGTH, features)

Creating training sequences
Creating validation sequences


In [31]:
gru_model = train_gru_model(train_sequences, train_targets, val_sequences, val_targets, features)

print("\nEvaluation of GRU model")
val_X_tensor = torch.FloatTensor(val_sequences).to(DEVICE)
val_y_tensor = torch.FloatTensor(val_targets).to(DEVICE).unsqueeze(1)
val_dataset = TensorDataset(val_X_tensor, val_y_tensor)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

gru_predictions, gru_targets = evaluate_gru_model(gru_model, val_loader)
gru_rmse = np.sqrt(mean_squared_error(gru_targets, gru_predictions))
print(f"\nGRU Model validation RMSE: {gru_rmse:.8f}")
print(f"Ridge Model validation RMSE: {ridge_rmse:.8f}")

Epoch 1/20: 100%|██████████| 1551/1551 [00:22<00:00, 69.57it/s]


Epoch 1: Train Loss = 0.00425470, Val Loss = 0.00002876


Epoch 2/20: 100%|██████████| 1551/1551 [00:22<00:00, 68.06it/s]


Epoch 2: Train Loss = 0.00006271, Val Loss = 0.00000937


Epoch 3/20: 100%|██████████| 1551/1551 [00:21<00:00, 71.64it/s]


...

Epoch 14: Train Loss = 0.00001264, Val Loss = 0.00000510


Epoch 15/20: 100%|██████████| 1551/1551 [00:20<00:00, 74.54it/s]


Epoch 15: Train Loss = 0.00001264, Val Loss = 0.00000509
Early stopping at epoch 15

Evaluation of GRU model


Evaluating: 100%|██████████| 802/802 [00:05<00:00, 154.19it/s]


GRU Model validation RMSE: 0.00224917
Ridge Model validation RMSE: 0.00224144


In [35]:
print("Generating GRU predictions...")
gru_results_df = predict_gru_test(gru_model, test, features, SEQ_LENGTH, batch_size=256)
test['prediction_gru'] = gru_results_df['prediction_gru']

# test = test.merge(gru_results_df, on='Id', how='left')
ridge_weight = 1.0 / ridge_rmse
gru_weight = 1.0 / gru_rmse
total_weight = ridge_weight + gru_weight

ridge_weight /= total_weight
gru_weight /= total_weight

test['prediction'] = ridge_weight * test['prediction_ridge'] + gru_weight * test['prediction_gru']
train_y_std = optimized_train['y'].std()
test['prediction'] = test['prediction'].clip(-3 * train_y_std, 3 * train_y_std)

t_original = pd.read_csv('/content/drive/MyDrive/quadeye-market-data-prediction-challenge/test.csv')
submission = t_original[['Id']].merge(test[['Id', 'prediction']], on='Id', how='left')
submission = submission.rename(columns={'prediction': 'y'})
submission.to_csv('submission.csv', index=False)

Generating GRU predictions...


Predicting test: 100%|██████████| 239/239 [00:27<00:00,  8.79it/s]
